# Demonstração Tapajós / TELEMAC

Os exemplos usam Culm e séries sintéticas explicitamente identificadas. Nenhum resultado é calibração do Tapajós. Execute a partir da pasta notebooks ou da raiz, com as dependências do projeto instaladas.

In [1]:
from pathlib import Path
import os, sys
ROOT = Path.cwd() if (Path.cwd() / 'backend').exists() else Path.cwd().parent
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))


In [2]:
from backend.api import cas_parser,slf_io,qsl_io,cli_io
p=ROOT/'data/culm'
params=cas_parser.parse((p/'t2d_culm.cas').read_text())
params.update({'NUMBER OF TIME STEPS':'20','GRAPHIC PRINTOUT PERIOD':'5','LISTING PRINTOUT PERIOD':'5'})
print(cas_parser.generate(params))

BOUNDARY CONDITIONS FILE = geo_culm.cli
GEOMETRY FILE = geo_culm.slf
PREVIOUS COMPUTATION FILE = ini_culm.slf
RESULTS FILE = r2d_culm.slf
LIQUID BOUNDARIES FILE = t2d_culm.qsl
TITLE = 'TELEMAC 2D : RIVER CULM'
COMPUTATION CONTINUED = YES
INITIAL TIME SET TO ZERO = YES
TIME STEP = 2.
NUMBER OF TIME STEPS = 20
GRAPHIC PRINTOUT PERIOD = 5
LISTING PRINTOUT PERIOD = 5
VARIABLES FOR GRAPHIC PRINTOUTS = 'U,V,S,B,H'
LAW OF BOTTOM FRICTION = 3
FRICTION COEFFICIENT = 30.
VELOCITY DIFFUSIVITY = 2.
TURBULENCE MODEL = 1
SOLVER ACCURACY = 1.E-4
IMPLICITATION FOR DEPTH = 1.
IMPLICITATION FOR VELOCITY = 1.
INITIAL CONDITIONS = 'CONSTANT ELEVATION'
INITIAL ELEVATION = 50.
PRESCRIBED FLOWRATES = 28.32 ;  0.
PRESCRIBED ELEVATIONS = 0.   ; 23.77
MASS-BALANCE = YES
TYPE OF ADVECTION = 1;5
SUPG OPTION = 1;0
MASS-LUMPING ON H = 1.
TIDAL FLATS = YES
OPTION FOR THE TREATMENT OF TIDAL FLATS = 1
MATRIX-VECTOR PRODUCT = 2
TREATMENT OF THE LINEAR SYSTEM = 2
SOLVER = 1



In [3]:
from backend.api import telemac_runner as runner
files={f.name:f.read_bytes() for f in p.iterdir() if f.suffix in ('.slf','.qsl','.cli') and not f.name.startswith('f2d')}
job=runner.prepare(params,files)
print('Entradas isoladas em',job['path'])

Entradas isoladas em /work/runs/ffb97e646a1142d29d5697722bdd66c3


Defina TELEMAC_EXECUTABLE ou TELEMAC_CONTAINER no ambiente antes de executar a próxima célula. O solver real será chamado.

In [4]:
await runner.run(job)
print(job['status'])
print('\n'.join(job['logs'][-20:]))
assert job['status']=='completed', 'Consulte o log e configure o solver'

completed
                     *************************************
                     *    END OF MEMORY ORGANIZATION:    *
                     *************************************

 CORRECT END OF RUN

 ELAPSE TIME :
                              0  SECONDS
... merging separated result files

... handling result files

        moving: r2d_culm.slf
... deleting working dir



My work is done




In [5]:
from scripts.postprocess import metrics,export_vtk
result=slf_io.read(job['path']/job['result'])
fields=metrics(result)
print(result.times,fields['speed'].max())
export_vtk(result,job['path']/'results.vtk',-1)

[ 0. 10. 20. 30. 40.] 1.5370447839733192
